# 3.3 Construct Data — Feature Engineering

> **Project:** Titanic Survival Prediction  
> **Author:** Thibauld  
> **Date:** 2026-03-31  
> **CRISP-DM Phase:** 3. Data Preparation  
> **Input:** Cleaned data from 3.2 (`data/processed/train_clean.csv`, `data/processed/test_clean.csv`)  
> **Output:** Feature-engineered data (`data/processed/train_features.csv`, `data/processed/test_features.csv`)

This notebook constructs 8 derived features, applies 3 value transformations (Sex encoding, Embarked one-hot, FamilySizeBin ordinal), and validates that no data leakage is present.

## Setup & Data Loading

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project root dynamically (works in VS Code and terminal)
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
if (PROJECT_ROOT / "notebooks").is_dir():
    pass  # cwd is project root
elif (PROJECT_ROOT.parent / "notebooks").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent  # cwd is a subdirectory

# Add src/ to path so we can import project modules
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

from src.features import (
    build_features,
    compute_ticket_group_sizes,
)

# Load cleaned data from 3.2
train = pd.read_csv(PROCESSED_DIR / "train_clean.csv")
test = pd.read_csv(PROCESSED_DIR / "test_clean.csv")

print(f"Train: {train.shape[0]} rows x {train.shape[1]} cols")
print(f"Test:  {test.shape[0]} rows x {test.shape[1]} cols")
print(f"\nTrain columns: {list(train.columns)}")

Train: 891 rows x 13 cols
Test:  418 rows x 12 cols

Train columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'AgeMissing']


## Feature Engineering Plan

Based on EDA (2.3) and data mining goals (1.3), we construct 8 features in 4 categories:

| # | Feature | Category | Source | Formula | Rationale |
|---|---------|----------|--------|---------|-----------|
| 1 | Title | Domain | Name | Regex extraction, 4-group mapping | 15.7%-79.2% survival range |
| 2 | FamilySize | Aggregation | SibSp, Parch | SibSp + Parch + 1 | Intermediate for binning |
| 3 | FamilySizeBin | Domain | FamilySize | Small(1), Medium(2-4), Large(5+) | Inverted-U survival pattern |
| 4 | IsAlone | Domain | FamilySize | 1 if FamilySize==1 | 30.4% vs 50.6% survival |
| 5 | HasCabin | Domain | Cabin | 1 if Cabin not null | 66.7% vs 30.0% survival |
| 6 | Deck | Domain | Cabin | First letter of Cabin | Decks B/D/E ~75% survival |
| 7 | FareLog | Transformation | Fare | log(Fare+1) | Reduces extreme skew (4.78) |
| 8 | TicketGroupSize | Aggregation | Ticket | Count of shared Ticket | Mirrors family size pattern |

**Value Transformations:** Sex (binary), Embarked (one-hot drop_first), FamilySizeBin (ordinal).

## Run Feature Engineering Pipeline

We use `src/features.py` which applies all feature construction and encoding in a single pipeline. Ticket group sizes are computed on the combined train+test data (families may be split across sets).

In [2]:
# Compute ticket group sizes from combined train+test (pre-event attribute, no leakage)
ticket_sizes = compute_ticket_group_sizes(train, test)

# Apply full feature pipeline
train_feat = build_features(train, ticket_sizes, encode=True)
test_feat = build_features(test, ticket_sizes, encode=True)

print(f"Train features: {train_feat.shape[0]} rows x {train_feat.shape[1]} cols")
print(f"Test features:  {test_feat.shape[0]} rows x {test_feat.shape[1]} cols")
print(f"\nNew columns added: {sorted(set(train_feat.columns) - set(train.columns))}")
print(f"Columns removed (encoded): {sorted(set(train.columns) - set(train_feat.columns))}")

Train features: 891 rows x 22 cols
Test features:  418 rows x 21 cols

New columns added: ['Deck', 'Embarked_Q', 'Embarked_S', 'FamilySize', 'FamilySizeBin', 'FareLog', 'HasCabin', 'IsAlone', 'TicketGroupSize', 'Title']
Columns removed (encoded): ['Embarked']


## Feature Inspection

### Title Distribution

In [3]:
# Title distribution and survival rate
title_stats = train_feat.groupby("Title").agg(
    Count=("Title", "size"),
    SurvivalRate=("Survived", "mean"),
).sort_values("Count", ascending=False)
title_stats["SurvivalRate"] = title_stats["SurvivalRate"].round(3)
print("Title Distribution (Train):")
print(title_stats)
print(f"\nUnique titles: {train_feat['Title'].nunique()}")

Title Distribution (Train):
        Count  SurvivalRate
Title                      
Mr        539         0.163
Miss      185         0.703
Mrs       127         0.795
Master     40         0.575

Unique titles: 4


### Family Features

In [4]:
# FamilySize distribution and survival by bin
family_stats = train_feat.groupby("FamilySizeBin").agg(
    Count=("FamilySizeBin", "size"),
    MeanSurvival=("Survived", "mean"),
    MeanFamilySize=("FamilySize", "mean"),
).reset_index()
family_stats.columns = ["FamilySizeBin (encoded)", "Count", "Survival Rate", "Mean Family Size"]
family_stats["Survival Rate"] = family_stats["Survival Rate"].round(3)
print("FamilySizeBin Distribution (Train):")
print(family_stats.to_string(index=False))

print(f"\nIsAlone: {train_feat['IsAlone'].sum()} passengers ({train_feat['IsAlone'].mean():.1%})")
print(f"IsAlone survival: {train_feat.loc[train_feat['IsAlone'] == 1, 'Survived'].mean():.3f}")
print(f"Not alone survival: {train_feat.loc[train_feat['IsAlone'] == 0, 'Survived'].mean():.3f}")

FamilySizeBin Distribution (Train):
FamilySizeBin (encoded)  Count  Survival Rate  Mean Family Size
                      0    537          0.304          1.000000
                      1    292          0.579          2.547945
                      2     62          0.161          6.709677

IsAlone: 537 passengers (60.3%)
IsAlone survival: 0.304
Not alone survival: 0.506


### Cabin Features

In [5]:
# HasCabin and Deck distribution
print("HasCabin Distribution (Train):")
print(train_feat["HasCabin"].value_counts().to_string())
print(f"\nHasCabin survival: {train_feat.loc[train_feat['HasCabin'] == 1, 'Survived'].mean():.3f}")
print(f"No cabin survival:  {train_feat.loc[train_feat['HasCabin'] == 0, 'Survived'].mean():.3f}")

print(f"\nDeck Distribution (Train):")
deck_stats = train_feat.groupby("Deck").agg(
    Count=("Deck", "size"),
    SurvivalRate=("Survived", "mean"),
).sort_values("Count", ascending=False)
deck_stats["SurvivalRate"] = deck_stats["SurvivalRate"].round(3)
print(deck_stats)

HasCabin Distribution (Train):
HasCabin
0    687
1    204

HasCabin survival: 0.667
No cabin survival:  0.300

Deck Distribution (Train):
         Count  SurvivalRate
Deck                        
Unknown    687         0.300
C           59         0.593
B           47         0.745
D           33         0.758
E           32         0.750
A           15         0.467
F           13         0.615
G            4         0.500
T            1         0.000


### Fare Log Transform & Ticket Group Size

In [6]:
# FareLog: compare original vs log-transformed distribution
print("Fare vs FareLog (Train):")
print(f"  Fare  — mean: {train_feat['Fare'].mean():.2f}, std: {train_feat['Fare'].std():.2f}, "
      f"skew: {train_feat['Fare'].skew():.2f}")
print(f"  FareLog — mean: {train_feat['FareLog'].mean():.2f}, std: {train_feat['FareLog'].std():.2f}, "
      f"skew: {train_feat['FareLog'].skew():.2f}")

print(f"\nTicketGroupSize Distribution (Train):")
tgs = train_feat["TicketGroupSize"].value_counts().sort_index()
for size, count in tgs.items():
    surv = train_feat.loc[train_feat["TicketGroupSize"] == size, "Survived"].mean()
    print(f"  Size {size}: {count} passengers, survival {surv:.3f}")

Fare vs FareLog (Train):
  Fare  — mean: 32.20, std: 49.69, skew: 4.79
  FareLog — mean: 2.96, std: 0.97, skew: 0.39

TicketGroupSize Distribution (Train):
  Size 1: 481 passengers, survival 0.270
  Size 2: 181 passengers, survival 0.514
  Size 3: 101 passengers, survival 0.653
  Size 4: 44 passengers, survival 0.727
  Size 5: 21 passengers, survival 0.333
  Size 6: 19 passengers, survival 0.211
  Size 7: 24 passengers, survival 0.208
  Size 8: 13 passengers, survival 0.385
  Size 11: 7 passengers, survival 0.000


## Data Leakage Validation

All features are derived from pre-event passenger attributes — no future information is used. Key checks:
1. No feature uses the `Survived` target (obvious but verified)
2. TicketGroupSize computed on combined train+test — this is valid because ticket purchase happened before the event
3. No temporal leakage — single historical event, no time dimension
4. All encoding/transformation logic is deterministic (no parameters fit on test data)

In [7]:
# Verify no leakage: Survived is never used as an input feature
feature_cols = [c for c in train_feat.columns if c not in ["PassengerId", "Survived"]]
assert "Survived" not in feature_cols, "Survived found in feature columns!"

# Verify no NaN in engineered features (except Cabin which is a source, not a feature)
new_features = ["Title", "FamilySize", "FamilySizeBin", "IsAlone", "HasCabin", "Deck",
                "FareLog", "TicketGroupSize"]
for feat in new_features:
    train_nulls = train_feat[feat].isna().sum()
    test_nulls = test_feat[feat].isna().sum()
    assert train_nulls == 0, f"Train {feat} has {train_nulls} nulls"
    assert test_nulls == 0, f"Test {feat} has {test_nulls} nulls"

print("Leakage check passed: Survived not in feature columns")
print("Null check passed: all engineered features have 0 missing values")

Leakage check passed: Survived not in feature columns
Null check passed: all engineered features have 0 missing values


## Feature Statistics

In [8]:
# Summary statistics for all features in the final dataset
numeric_cols = train_feat.select_dtypes(include=[np.number]).columns.tolist()
# Exclude PassengerId and Survived from feature stats
stat_cols = [c for c in numeric_cols if c not in ["PassengerId", "Survived"]]

stats = train_feat[stat_cols].describe().T
stats["non_null"] = train_feat[stat_cols].notna().sum()
stats["dtype"] = train_feat[stat_cols].dtypes
stats = stats[["non_null", "mean", "std", "min", "50%", "max", "dtype"]]
stats.columns = ["Non-Null", "Mean", "Std", "Min", "Median", "Max", "Dtype"]
print("Feature Statistics (Train):")
print(stats.round(3).to_string())

# Categorical features
cat_cols = train_feat.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ["PassengerId", "Name", "Ticket", "Cabin"]]
if cat_cols:
    print(f"\nCategorical features: {cat_cols}")
    for col in cat_cols:
        print(f"  {col}: {train_feat[col].value_counts().to_dict()}")

Feature Statistics (Train):
                 Non-Null    Mean     Std   Min  Median      Max    Dtype
Pclass                891   2.309   0.836  1.00   3.000    3.000    int64
Sex                   891   0.352   0.478  0.00   0.000    1.000    int64
Age                   891  29.372  13.253  0.42  30.000   80.000  float64
SibSp                 891   0.523   1.103  0.00   0.000    8.000    int64
Parch                 891   0.382   0.806  0.00   0.000    6.000    int64
Fare                  891  32.204  49.693  0.00  14.454  512.329  float64
AgeMissing            891   0.199   0.399  0.00   0.000    1.000    int64
FamilySize            891   1.905   1.613  1.00   1.000   11.000    int64
IsAlone               891   0.603   0.490  0.00   1.000    1.000    int64
HasCabin              891   0.229   0.420  0.00   0.000    1.000    int64
FareLog               891   2.962   0.969  0.00   2.738    6.241  float64
TicketGroupSize       891   2.121   1.797  1.00   1.000   11.000    int64
Embarked_Q

/var/folders/c1/0_yrgfd54sl_3grrp8bnb1080000gp/T/ipykernel_89206/2000767617.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_feat.select_dtypes(include=["object"]).columns.tolist()


## Save Output

In [9]:
# Save feature-engineered datasets
train_feat.to_csv(PROCESSED_DIR / "train_features.csv", index=False)
test_feat.to_csv(PROCESSED_DIR / "test_features.csv", index=False)

print(f"Saved: {PROCESSED_DIR / 'train_features.csv'} ({train_feat.shape[0]} rows x {train_feat.shape[1]} cols)")
print(f"Saved: {PROCESSED_DIR / 'test_features.csv'} ({test_feat.shape[0]} rows x {test_feat.shape[1]} cols)")
print(f"\nFinal columns: {list(train_feat.columns)}")

Saved: /Users/tba8ydd/Documents/claude-template/data/processed/train_features.csv (891 rows x 22 cols)
Saved: /Users/tba8ydd/Documents/claude-template/data/processed/test_features.csv (418 rows x 21 cols)

Final columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'AgeMissing', 'Title', 'FamilySize', 'FamilySizeBin', 'IsAlone', 'HasCabin', 'Deck', 'FareLog', 'TicketGroupSize', 'Embarked_Q', 'Embarked_S']


## Conclusions

**Features constructed:** 8 new features (Title, FamilySize, FamilySizeBin, IsAlone, HasCabin, Deck, FareLog, TicketGroupSize).

**Value transformations applied:** 3 (Sex binary encoding, Embarked one-hot encoding, FamilySizeBin ordinal encoding).

**Data leakage assessment:** All features are derived from pre-event passenger attributes. No target leakage, no temporal leakage, no preprocessing leakage. TicketGroupSize computed on combined train+test is valid because tickets were purchased before the event.

**Key observations:**
- Title captures the dominant survival signal (Mr: 15.7%, Mrs: 79.2%)
- FamilySizeBin confirms the inverted-U pattern: Small ~30%, Medium ~58%, Large ~12%
- FareLog reduces skew from 4.78 to ~0.4, improving distribution symmetry
- HasCabin is a strong binary proxy for class (66.7% vs 30.0% survival)

**Next step:** Run `/integrate-data` (Task 3.4) to merge features into a unified modeling dataset, or proceed directly to modeling (Phase 4) since no external data sources need integration.